# 🎨 PHASE 3: Streamlit Web-UI, HTML Embed & Dynamic Provisioner
**Projekt: Mai_AI (MaiOmni) — Javes / OpenHuman-Style Frontend**

Hier implementieren wir das **Dynamic User Container System (Javes / OpenHuman-Alternative)**. Jedes Mal, wenn sich ein Benutzer über das Auth-Gateway anmeldet, erstellt unser System dynamisch eine komplett isolierte Instanz, die eine Streamlit-Webseite hostet und nahtlos für den Endnutzer darstellt.

--- 
### 🎯 Ziele dieser Phase:
1. Bauen/Verifizieren des Frontend-Docker-Images (`python:3.11-slim` oder dediziertes `mai_ai_image`).
2. Programmatisches Provisionieren eines Test-Benutzer-Containers.
3. Integration der Web-Oberfläche (Iframe/Embed-Verhalten).
4. Dynamic Routing-Checks.

---

### 🏗️ Schritt 1: Das Benutzer-Image verifizieren
Jeder Benutzer-Container nutzt eine schlanke Python-Umgebung, auf der Streamlit und die Verbindung zu Ollama laufen.

In [1]:
import docker
client = docker.from_env()

print("Hole Standard-Image 'python:3.11-slim' (falls nicht vorhanden)...")
try:
    image = client.images.pull("python:3.11-slim")
    print(f"[✓] Standard-Image bereit: {image.tags}")
except Exception as e:
    print(f"[!] Fehler beim Herunterladen des Images: {e}")

DockerException: Error while fetching server API version: 503 Server Error for http+docker://localnpipe/version: Service Unavailable ("Docker Desktop is unable to start")

### ⚡ Schritt 2: Dynamische Benutzer-Provisionierung testen
Wir rufen das `MultiUserManager`-Modul auf, um virtuell einen Test-User zu provisionieren. Dies erzeugt die Traefik-Labels im Container, die auf `X-Forwarded-User` hören.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.docker_py.docker_user_manager import MultiUserManager

manager = MultiUserManager()
test_user = "testuser@mai-ai.local"

print(f"Erzeuge isolierte Sandbox für: {test_user}...")
container = manager.provision_user(test_user)

if container:
    print(f"[✓] Container erfolgreich gestartet!")
    print(f"ID: {container.short_id}")
    print(f"Name: {container.name}")
    print(f"Labels für Traefik Routing:")
    for k, v in container.labels.items():
        if "traefik" in k:
            print(f"  {k} = {v}")
    # Bereinigung nach dem Test
    print("\nRäume Test-Container auf...")
    container.stop()
    container.remove()
    print("[✓] Test-Umgebung sauber aufgeräumt.")
else:
    print("[!] Fehler bei der Provisionierung.")

Erzeuge isolierte Sandbox für: testuser@mai-ai.local...
[✓] Container erfolgreich gestartet!
ID: 72516352472a
Name: ai_user_testuser_mai-ai_local
Labels für Traefik Routing:
  traefik.enable = true
  traefik.http.routers.user_testuser_mai-ai_local.entrypoints = web
  traefik.http.routers.user_testuser_mai-ai_local.priority = 20
  traefik.http.routers.user_testuser_mai-ai_local.rule = (Host(`platform.local`) || Host(`mai-ai.duckdns.org`)) && Header(`X-Forwarded-User`, `testuser@mai-ai.local`)
  traefik.http.services.user_testuser_mai-ai_local.loadbalancer.server.port = 8000

Räume Test-Container auf...
[✓] Test-Umgebung sauber aufgeräumt.


### 🖥️ Schritt 3: HTML Iframe Embedding Prinzip (NextToken / Javes Style)
Das UI-Dashboard bindet den User-Container nahtlos als HTML-Iframe ein. Der Auth-Proxy sorgt dafür, dass nur der angemeldete Nutzer seinen eigenen Container sieht.

```html
<!-- Das Prinzip wie es im Browser gerendert wird -->
<iframe 
    src="http://platform.local/"
    style="width: 100%; height: 100vh; border: none;"
    allow="clipboard-read; clipboard-write">
</iframe>
```

### 🔄 Was kommt als Nächstes?
Deine dynamische Benutzeroberfläche und die containerisierte Sandbox stehen!

Fahre fort mit dem letzten und wichtigsten Sicherheits-Knotenpunkt:
👉 **[04_sicherheit.ipynb](file:notebooks/04_sicherheit.ipynb)** um das System absolut bruchsicher zu machen.